### ALGORITMO GENÉTICO com Mineração de Dados

O objetivo é isolar o ganho que a mineração traz ao AG especificamente, comparando com o AG puro. O K-Means atua em:

- Inicialização Inteligente:
Ao invés de gerar toda a população inicial aleatoriamente, metade das soluções é gerada com bias nos centróides encontrados via K-Means aplicado sobre as coordenadas de todos os pontos de demanda ponderados pelo peso populacional. Isso concentra ambulâncias nas regiões de maior demanda desde o início.

- Crossover guiado:
Ao gerar filhos, além do crossover padrão, genes próximos de centroides de elite (atualizados periodicamente) têm maior probabilidade de ser preservados no filho - o algoritmo aprende quais regiões são promissoras.

- Mutação dirigida:
    Em vex de mutar para uma posição completamente aleatória, a mutação sorteia entre posições próximas a centroides com probabilidade proporcional à sua cobertura potencial.


#### IMPORTS

In [2]:
import random
import math
import time
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
from sklearn.cluster import KMeans

from baseline.POSSIBLE import DISTRICTS_POINTS as DP

### Parâmetros do problema

In [3]:
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336
BONUS_TYPE_A = 1.20
TOTAL_AMBUS = NUMBER_AMBUS_TYPE_A + NUMBER_AMBUS_TYPE_B

### Parâmetros do GA

In [4]:
POPULATION_SIZE = 200 
MAX_GENERATIONS = 500
PCT_ELITISM = 0.30 # preserva 30% dos melhores
PCT_RANDOM = 0.20 # 20% aleatórios para diversidade
MUTATION_RATE = 0.05
DISASTER_EVERY = 50 # a cada 50 geracoes sem melhora, gera uma catastrofe
PCT_DISASTER_DEATH = 0.40 # % da população que morre na catastrofe

### Parâmetros da Hibridização

In [5]:
PCT_BIASED_INIT      = 0.50    # % da pop. inicial gerada com bias de centroides
DM_UPDATE_EVERY      = 50      # a cada N gerações atualiza centroides de elite
ELITE_POOL_SIZE      = 20      # soluções elite para minerar
N_CLUSTERS           = TOTAL_AMBUS
BIAS_RADIUS          = 0.015   # raio de influência dos centroides
GUIDED_CROSSOVER_PROB = 0.60   # prob. de usar crossover guiado vs padrão
GUIDED_MUTATION_PROB  = 0.70   # prob. de usar mutação dirigida vs aleatória

### Estruturas dos Dados

In [6]:
NUMBER_LOCATIONS = len(DP)
_keys = list(DP.keys())

COORDS_X = np.array([DP[k][0] for k in _keys])
COORDS_Y = np.array([DP[k][1] for k in _keys])
WEIGHTS = np.array([float(DP[k][2]) if str(DP[k][2]) != "nan" else 0.0 for k in _keys])

COORDS = np.column_stack([COORDS_X, COORDS_Y])
TOTAL_DEMAND = float(WEIGHTS.sum())

@dataclass
class Solution:
    xA: List[int] = field(default_factory = lambda: [0] * NUMBER_LOCATIONS)
    xB: List[int] = field(default_factory = lambda: [0] * NUMBER_LOCATIONS)
    non_zeros_A: List[int] = field(default_factory=list)
    non_zeros_B: List[int] = field(default_factory=list)
    fitness: float = 0.0
    coverage: float = 0.0

    def __lt__(self, other: Solution): return self.fitness < other.fitness
    def __gt__(self, other: Solution): return self.fitness > other.fitness
    def __le__(self, other: Solution): return self.fitness <= other.fitness
    def __ge__(self, other: Solution): return self.fitness >= other.fitness


### Pré-computação da matriz de distâncias

In [7]:
def build_distance_matrix() -> np.ndarray:
    diff_x = COORDS_X[:, None] - COORDS_X[None, :]
    diff_y = COORDS_Y[:, None] - COORDS_Y[None, :]
    return np.sqrt(diff_x**2 + diff_y**2)

### Avaliação

In [8]:
def evaluate_solution(sol: Solution, dist_matrix: np.ndarray) -> None:
    covered_A: Set[int] = set()
    covered_B: Set[int] = set()

    for a in sol.non_zeros_A:
        covered_A.update(np.where(dist_matrix[a] <= RAIO)[0].tolist())
    for b in sol.non_zeros_B:
        covered_B.update(np.where(dist_matrix[b] <= RAIO)[0].tolist())

    fitness = float((WEIGHTS[list(covered_A)] * BONUS_TYPE_A).sum())
    only_B  = covered_B - covered_A
    if only_B:
        fitness += float(WEIGHTS[list(only_B)].sum())

    all_covered    = covered_A | covered_B
    sol.fitness    = fitness
    sol.coverage   = float(WEIGHTS[list(all_covered)].sum() / TOTAL_DEMAND * 100.0)


def coverage_percentage(sol: Solution) -> float:
    return sol.coverage

### Reparo

In [9]:
def repair(sol: Solution) -> None:
    sol.non_zeros_A = [i for i in range(NUMBER_LOCATIONS) if sol.xA[i] == 1]
    sol.non_zeros_B = [i for i in range(NUMBER_LOCATIONS) if sol.xB[i] == 1]

    # Resolve sobreposição: TypeA tem prioridade
    overlap = set(sol.non_zeros_A) & set(sol.non_zeros_B)
    for idx in overlap:
        sol.xB[idx] = 0
        sol.non_zeros_B.remove(idx)

    # Ajusta TypeA
    while len(sol.non_zeros_A) > NUMBER_AMBUS_TYPE_A:
        rm = random.choice(sol.non_zeros_A)
        sol.xA[rm] = 0
        sol.non_zeros_A.remove(rm)

    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)
    while len(sol.non_zeros_A) < NUMBER_AMBUS_TYPE_A:
        idx = random.randrange(NUMBER_LOCATIONS)
        if idx not in occupied:
            sol.xA[idx] = 1
            sol.non_zeros_A.append(idx)
            occupied.add(idx)

    # Ajusta TypeB
    while len(sol.non_zeros_B) > NUMBER_AMBUS_TYPE_B:
        rm = random.choice(sol.non_zeros_B)
        sol.xB[rm] = 0
        sol.non_zeros_B.remove(rm)

    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)
    while len(sol.non_zeros_B) < NUMBER_AMBUS_TYPE_B:
        idx = random.randrange(NUMBER_LOCATIONS)
        if idx not in occupied:
            sol.xB[idx] = 1
            sol.non_zeros_B.append(idx)
            occupied.add(idx)


### Mineração: K-Means 

In [10]:
def compute_centroids_from_demand() -> np.ndarray:
    """
    Camada 1: K-Means sobre todos os pontos de demanda ponderados
    pelo peso populacional. Usado na inicialização.
    """
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
    km.fit(COORDS, sample_weight=WEIGHTS)
    return km.cluster_centers_


def compute_centroids_from_elite(elite_pool: List[Solution]) -> np.ndarray:
    """
    Camadas 2 e 3: K-Means sobre as posições das soluções elite.
    Atualizado periodicamente durante a evolução.
    """
    all_positions = []
    for sol in elite_pool:
        for p in sol.non_zeros_A + sol.non_zeros_B:
            all_positions.append([COORDS_X[p], COORDS_Y[p]])

    if len(all_positions) < N_CLUSTERS:
        return compute_centroids_from_demand()

    pts = np.array(all_positions)
    km  = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
    km.fit(pts)
    return km.cluster_centers_


def locations_near_centroids(centroids: np.ndarray) -> List[int]:
    """Retorna índices de localização dentro de BIAS_RADIUS de algum centroide."""
    biased = set()
    for cx, cy in centroids:
        dists = np.sqrt((COORDS_X - cx)**2 + (COORDS_Y - cy)**2)
        biased.update(np.where(dists <= BIAS_RADIUS)[0].tolist())
    return list(biased)


def closest_to_centroid(centroid: np.ndarray, exclude: Set[int]) -> int:
    """Retorna o índice de localização mais próximo do centroide, excluindo ocupados."""
    dists = np.sqrt((COORDS_X - centroid[0])**2 + (COORDS_Y - centroid[1])**2)
    order = np.argsort(dists)
    for idx in order:
        if int(idx) not in exclude:
            return int(idx)
    return random.randrange(NUMBER_LOCATIONS)

### Criação de Soluções

In [11]:
def create_solution_biased(centroids: np.ndarray) -> Solution:
    """
    Gera solução com ambulâncias posicionadas próximas aos centroides
    de demanda. Cada centroide sugere uma localização candidata.
    """
    sol      = Solution()
    occupied: Set[int] = set()

    # Embaralha centroides para não fixar sempre a mesma ordem
    centroid_order = list(range(len(centroids)))
    random.shuffle(centroid_order)

    # Aloca TypeA nos primeiros centroides
    for i in range(NUMBER_AMBUS_TYPE_A):
        c   = centroids[centroid_order[i]]
        idx = closest_to_centroid(c, occupied)
        sol.xA[idx] = 1
        sol.non_zeros_A.append(idx)
        occupied.add(idx)

    # Aloca TypeB nos centroides seguintes
    for i in range(NUMBER_AMBUS_TYPE_A, TOTAL_AMBUS):
        c   = centroids[centroid_order[i]]
        idx = closest_to_centroid(c, occupied)
        sol.xB[idx] = 1
        sol.non_zeros_B.append(idx)
        occupied.add(idx)

    return sol


def create_solution_random() -> Solution:
    """Solução totalmente aleatória (sem bias)."""
    sol      = Solution()
    pool     = list(range(NUMBER_LOCATIONS))
    chosen_A = random.sample(pool, NUMBER_AMBUS_TYPE_A)
    for idx in chosen_A:
        sol.xA[idx] = 1
        sol.non_zeros_A.append(idx)
    remaining = [i for i in pool if i not in chosen_A]
    chosen_B  = random.sample(remaining, NUMBER_AMBUS_TYPE_B)
    for idx in chosen_B:
        sol.xB[idx] = 1
        sol.non_zeros_B.append(idx)
    return sol

### Seleção

In [12]:
def roulette_select(population: List[Solution]) -> Solution:
    total = sum(s.fitness for s in population)
    if total == 0:
        return random.choice(population)
    pick       = random.uniform(0, total)
    cumulative = 0.0
    for s in population:
        cumulative += s.fitness
        if cumulative >= pick:
            return s
    return population[-1]

### Crossover

In [13]:
def crossover_guided(base: Solution, guide: Solution,
                     biased_locs: Set[int],
                     dist_matrix: np.ndarray) -> Solution:
    """
    Crossover padrão com regra adicional: quando só o guide tem um gene
    e essa posição está próxima de um centroide de elite, a probabilidade
    de preservá-la aumenta de 0.5 para 0.8.
    """
    child = Solution()

    for vec in ['A', 'B']:
        base_x  = base.xA  if vec == 'A' else base.xB
        guide_x = guide.xA if vec == 'A' else guide.xB
        child_x = child.xA if vec == 'A' else child.xB

        for i in range(NUMBER_LOCATIONS):
            if base_x[i] == guide_x[i]:
                child_x[i] = base_x[i]
            elif base_x[i] == 1:
                # Só base tem → sempre preserva (gene de maior fitness)
                child_x[i] = 1
            else:
                # Só guide tem → prob. maior se próximo de centroide
                prob = 0.80 if i in biased_locs else 0.50
                child_x[i] = 1 if random.random() < prob else 0

    repair(child)
    evaluate_solution(child, dist_matrix)
    return child


def crossover_standard(base: Solution, guide: Solution,
                       dist_matrix: np.ndarray) -> Solution:
    """Crossover padrão do AG sem influência dos centroides."""
    child = Solution()

    for vec in ['A', 'B']:
        base_x  = base.xA  if vec == 'A' else base.xB
        guide_x = guide.xA if vec == 'A' else guide.xB
        child_x = child.xA if vec == 'A' else child.xB

        for i in range(NUMBER_LOCATIONS):
            if base_x[i] == guide_x[i]:
                child_x[i] = base_x[i]
            elif base_x[i] == 1:
                child_x[i] = 1
            else:
                child_x[i] = random.randint(0, 1)

    repair(child)
    evaluate_solution(child, dist_matrix)
    return child

### MUTAÇÃO

In [14]:
def mutate(sol: Solution, dist_matrix: np.ndarray,
           biased_locs: List[int]) -> None:
    """
    Com prob. MUTATION_RATE, troca uma ambulância de posição.
    Se usar mutação dirigida (GUIDED_MUTATION_PROB), a nova posição
    é sorteada entre os locais próximos de centroides de elite.
    Caso contrário, posição aleatória (comportamento padrão do AG).
    """
    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)

    def pick_new_position(exclude: Set[int]) -> int:
        if biased_locs and random.random() < GUIDED_MUTATION_PROB:
            candidates = [l for l in biased_locs if l not in exclude]
            if candidates:
                return random.choice(candidates)
        # Fallback: aleatório
        for _ in range(50):
            idx = random.randrange(NUMBER_LOCATIONS)
            if idx not in exclude:
                return idx
        return random.randrange(NUMBER_LOCATIONS)

    if random.random() < MUTATION_RATE and sol.non_zeros_A:
        old_pos = random.choice(sol.non_zeros_A)
        new_pos = pick_new_position(occupied - {old_pos})
        if new_pos != old_pos and new_pos not in occupied:
            sol.xA[old_pos] = 0
            sol.non_zeros_A.remove(old_pos)
            occupied.discard(old_pos)
            sol.xA[new_pos] = 1
            sol.non_zeros_A.append(new_pos)
            occupied.add(new_pos)

    if random.random() < MUTATION_RATE and sol.non_zeros_B:
        old_pos = random.choice(sol.non_zeros_B)
        new_pos = pick_new_position(occupied - {old_pos})
        if new_pos != old_pos and new_pos not in occupied:
            sol.xB[old_pos] = 0
            sol.non_zeros_B.remove(old_pos)
            occupied.discard(old_pos)
            sol.xB[new_pos] = 1
            sol.non_zeros_B.append(new_pos)
            occupied.add(new_pos)

    evaluate_solution(sol, dist_matrix)

### Catástrofe

In [15]:
def disaster(population: List[Solution],
             centroids: np.ndarray,
             dist_matrix: np.ndarray) -> List[Solution]:
    n_die     = int(PCT_DISASTER_DEATH * len(population))
    survivors = population[n_die:]   # mantém os melhores (ordenado crescente)
    for i in range(n_die):
        # Metade dos novos ainda usa bias de centroides
        if i < n_die // 2:
            s = create_solution_biased(centroids)
        else:
            s = create_solution_random()
        evaluate_solution(s, dist_matrix)
        survivors.append(s)
    return survivors


### Função Principal

In [16]:
def run(seed: int = None, verbose: bool = False) -> Tuple[Solution, List[float]]:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    dist_matrix = build_distance_matrix()

    # ── Camada 1: Inicialização com centroides de demanda
    demand_centroids = compute_centroids_from_demand()
    elite_pool: List[Solution] = []
    biased_locs: List[int]    = locations_near_centroids(demand_centroids)
    current_centroids          = demand_centroids

    population = []
    n_biased   = int(PCT_BIASED_INIT * POPULATION_SIZE)
    for i in range(POPULATION_SIZE):
        s = (create_solution_biased(demand_centroids)
             if i < n_biased else create_solution_random())
        evaluate_solution(s, dist_matrix)
        population.append(s)
    population.sort()

    best = max(population, key=lambda s: s.fitness)
    history: List[float] = [best.coverage]
    gens_no_improve = 0

    for gen in range(MAX_GENERATIONS):

        # ── Atualização de centroides de elite (Camadas 2 e 3)
        if gen > 0 and gen % DM_UPDATE_EVERY == 0 and elite_pool:
            current_centroids = compute_centroids_from_elite(elite_pool)
            biased_locs       = locations_near_centroids(current_centroids)
            if verbose:
                print(f"  [DM] Gen {gen}: {len(biased_locs)} locais promissores")

        biased_set = set(biased_locs)
        new_pop    = []

        # Elitismo
        n_elite = int(PCT_ELITISM * POPULATION_SIZE)
        new_pop.extend(population[-n_elite:])

        # Aleatórios
        n_random = int(PCT_RANDOM * POPULATION_SIZE)
        for _ in range(n_random):
            s = create_solution_random()
            evaluate_solution(s, dist_matrix)
            new_pop.append(s)

        # Filhos
        n_children = POPULATION_SIZE - n_elite - n_random
        for _ in range(n_children):
            p1    = roulette_select(population)
            p2    = roulette_select(population)
            base  = p1 if p1.fitness >= p2.fitness else p2
            guide = p2 if base is p1 else p1

            # Camada 2: crossover guiado ou padrão
            if biased_locs and random.random() < GUIDED_CROSSOVER_PROB:
                child = crossover_guided(base, guide, biased_set, dist_matrix)
            else:
                child = crossover_standard(base, guide, dist_matrix)

            # Camada 3: mutação dirigida
            mutate(child, dist_matrix, biased_locs)
            new_pop.append(child)

        new_pop.sort()
        population = new_pop

        gen_best = population[-1]
        if gen_best.fitness > best.fitness:
            best = gen_best
            gens_no_improve = 0
        else:
            gens_no_improve += 1

        # Atualiza pool de elites
        elite_pool.append(gen_best)
        elite_pool.sort(reverse=True)
        if len(elite_pool) > ELITE_POOL_SIZE:
            elite_pool = elite_pool[:ELITE_POOL_SIZE]

        history.append(best.coverage)

        if verbose and gen % 50 == 0:
            print(f"  AG-H Gen {gen:4d} | Cobertura: {best.coverage:.2f}%")

        if gens_no_improve >= DISASTER_EVERY:
            population = disaster(population, current_centroids, dist_matrix)
            population.sort()
            gens_no_improve = 0

    return best, history


if __name__ == "__main__":
    print("=== AG Híbrido (K-Means em 3 camadas) ===")
    t0 = time.time()
    best_sol, hist = run(seed=42, verbose=True)
    elapsed = time.time() - t0
    print(f"\nMelhor cobertura: {best_sol.coverage:.2f}%")
    print(f"Tempo: {elapsed:.1f}s")
    print(f"Posições TypeA: {best_sol.non_zeros_A}")
    print(f"Posições TypeB: {best_sol.non_zeros_B}")


=== AG Híbrido (K-Means em 3 camadas) ===
  AG-H Gen    0 | Cobertura: 32.41%
  [DM] Gen 50: 1361 locais promissores
  AG-H Gen   50 | Cobertura: 39.65%
  [DM] Gen 100: 1397 locais promissores
  AG-H Gen  100 | Cobertura: 39.67%
  [DM] Gen 150: 1359 locais promissores
  AG-H Gen  150 | Cobertura: 40.13%
  [DM] Gen 200: 1344 locais promissores
  AG-H Gen  200 | Cobertura: 40.15%
  [DM] Gen 250: 1358 locais promissores
  AG-H Gen  250 | Cobertura: 40.24%
  [DM] Gen 300: 1360 locais promissores
  AG-H Gen  300 | Cobertura: 40.47%
  [DM] Gen 350: 1359 locais promissores
  AG-H Gen  350 | Cobertura: 40.47%
  [DM] Gen 400: 1359 locais promissores
  AG-H Gen  400 | Cobertura: 40.47%
  [DM] Gen 450: 1359 locais promissores
  AG-H Gen  450 | Cobertura: 40.47%

Melhor cobertura: 40.47%
Tempo: 29.1s
Posições TypeA: [1976]
Posições TypeB: [473, 1228, 1301, 2183]
